# ROSP-7: System delivered image quality (sDIQ)

PTH Tier 1 metric **ROSP-7** - {{ params.start_dayobs }} to {{ params.end_dayobs }} (dayObs, inclusive)

- Jira epic: [ROSP-7](https://rubinobs.atlassian.net/browse/ROSP-7)
- Confluence metric page: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906933804](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906933804)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Elana Urbach |
| Owner | Sandrine Thomas |
| Category | 2 - Data Production |
| Layer | 2 |

Method version: `rosp-07-v1.0` (draft until the tag is cut; the frozen commit SHA is recorded on the Confluence page). Changes to the method go through the PTH Method Change Process: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501
The upstream versions the method is pinned to (summit_utils `w.2026.44` for the effective wavelengths and for the guider
tomographic seeing module, its layer coefficients and outer scale; the Rapid Analysis package versions the ConsDB records
for the PSF and guider columns; the ESS CSC behind RINGSS) are listed and checked against the kernel and the data in the
Provenance cell. A change in any of them is a Method Change.

**Method.** The system contribution to the delivered image quality is what is left of the delivered PSF FWHM
once the atmosphere is removed in quadrature. For every LSSTCam science visit of the selected programs:

* the **delivered PSF FWHM** is `psf_sigma_median` × 2√(2 ln 2) × 0.2″/pixel from `cdb_lsstcam.visit1_quicklook`,
  exactly as in ROSP-2 and **as delivered** (no zenith / 500 nm correction);
* the **atmospheric seeing** is taken from an independent monitor and expressed in the same frame as the PSF,
  i.e. at the airmass $X$ and band of the visit. Monitors that report at zenith and 500 nm are de-corrected with the
  inverse of the ROSP-2 correction, $\theta_\mathrm{obs} = \theta_\mathrm{z,500}\,X^{0.6}\,(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{-0.2}$,
  with the same effective wavelengths as ROSP-2. Three monitors are used, each giving its own sDIQ:
  1. **RINGSS** scintillation seeing (`fwhmScintillation`, Kolmogorov seeing at zenith / 500 nm). Per visit, the per-visit mean already
     in the ConsDB (`efd_lsstcam.visit1_efd.mt_ringss_fwhm_scintillation`) is used when it exists; otherwise the median of the
     EFD measurements (`fwhmScintillation` of `lsst.sal.ESS.logevent_ringssMeasurement`, salIndex 304) within
     ± `ringss_window_minutes` of the visit midpoint, when the window holds any. The source used is recorded per visit
     (`ringss_src`); `ringss_source` can restrict it to one of the two.
  2. **DIMM** (`cdb_lsstcam.visit1.dimm_seeing`, Kolmogorov seeing at zenith / 500 nm). The DIMM is only used on visits
     with an outside wind speed (`visit1.wind_speed`) of at least `dimm_min_wind_speed` = {{ params.dimm_min_wind_speed }} m/s, below which it is known to be unreliable.
  3. **Guider** free-atmosphere seeing: the mid (0.5 - 1.5 km) and high (> 1.5 km) layers of the guider tomographic
     seeing, $\sqrt{\mathtt{guider\_mid\_layer\_seeing}^2 + \mathtt{guider\_free\_seeing}^2}$ from `visit1_quicklook`, without the ground layer.
     The producer (`lsst.summit.utils.guiders.seeing`) derives these from the guider-star centroid motion along the line of sight and scales
     them to the exposure wavelength, so they are already in the frame of the visit and are used as they are. The producer's
     own von Kármán model uses $L_0$ = 25 m (the `fwhmVonkarman` default, recorded in `PINNED`), not `outer_scale_m`, so part of
     any difference between the guider and the other two monitors can come from the outer scale alone.

RINGSS and the DIMM both report the Fried parameter as a Kolmogorov FWHM, $0.98\,\lambda/r_0$. The scintillation
measurement itself is insensitive to the outer scale, but the long-exposure FWHM that a given $r_0$ produces on the 8.4 m
mirror is not, so both are converted the same way: corrected for a finite outer scale $L_0$ = `outer_scale_m` (von Kármán,
Tokovinin 2002), with the Fried parameter scaled to the visit **before** the correction:
$r_0 = 0.98 \cdot 500\,\mathrm{nm} / \theta_\mathrm{monitor}$, $r_0(\lambda, X) = r_0\,(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{1.2}\,X^{-0.6}$,
$\theta_\mathrm{obs} = \dfrac{0.98\,\lambda_\mathrm{eff}}{r_0(\lambda, X)}\sqrt{1 - 2.18\,(r_0(\lambda, X)/L_0)^{0.356}}$.
The Kolmogorov part of this is the $X^{0.6}(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{-0.2}$ de-correction (the inverse of the ROSP-2
correction); the outer-scale factor is evaluated at the band and airmass of the visit rather than at zenith / 500 nm (a stronger
reduction in the red). `outer_scale_m` = 0 subtracts the Kolmogorov FWHM instead.

Per visit the squared system term is $\mathrm{sys}^2 = \mathrm{FWHM}^2 - \theta_\mathrm{atm,obs}^2$ (arcsec²), which is noisy and may be
negative. The metric for any set of visits is

$$ \mathrm{sDIQ} = \sqrt{\operatorname{median}(\mathrm{sys}^2)}, $$

the median taken before the square root so that negative visits are kept. A negative median is shown as the signed
square root. The sDIQ is quoted with its 25-75 % range in brackets: the signed square roots of the 25th and 75th percentiles of the per-visit $\mathrm{sys}^2$. The range shows how the visits are spread around the sDIQ; it is not the uncertainty of the sDIQ. The whiskers in the figures (nightly series, sanity checks) show the same range over the visits of a night or of a bin, and for the FWHM and the monitor seeing in the nightly series the 25th to 75th percentiles of the per-visit values.

**Selection.** The ROSP-2 selection: visits need a PSF measurement, at least {{ params.min_psf_stars }} PSF stars
(`n_psf_star_total`, the `min_psf_stars` parameter), an airmass and a band, and `filter_change_close_loop` visits are dropped.
Each monitor's sDIQ then uses the selected visits that have a finite, positive value from that monitor; the only monitor-specific cut is the DIMM wind-speed one.

Everything is given per monitor, for all bands together and per band: the sDIQ with its 25-75 % range and the spread of $\mathrm{sys}^2$ (table),
night by night (nightly sDIQ with the 25-75 % range of the per-visit term) and as histograms of $\mathrm{sys}^2$.

The headline values are the all-band sDIQ for the RINGSS and DIMM monitors. As of late 2026, the guider data is unreliable, the RINGSS went offline in mid-June, and the DIMM went offline in mid-July, so the survey sDIQ measurement is really based on 9 nights between 29 June and 13 July.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch and the standard science blocks.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = 20261231  # Last dayObs (inclusive)
science_programs = "407, 408, 416, 417, 419, 421"  # bare numbers become BLOCK-<n>; empty = all programs
min_psf_stars = (
    50  # drop visits with fewer PSF stars (n_psf_star_total) than this; 0 = no cut
)
outer_scale_m = 30  # outer scale L0 (m) for the von Karman correction of RINGSS and the DIMM; 0 = none
dimm_min_wind_speed = (
    4.0  # DIMM only used on visits with wind_speed at or above this (m/s); 0 = no cut
)
ringss_source = "both"  # "both": efd_lsstcam.visit1_efd per-visit mean, else the EFD median within the window; "consdb" or "efd": that source only
ringss_window_minutes = (
    5  # EFD RINGSS measurements within +/- this of the visit midpoint are medianed
)
efd_name = "usdf_efd"  # lsst_efd_client instance name
use_cache = False  # cache the ConsDB/EFD pulls as parquet under ./cache (ignored on Times Square)

not_times_square = True

In [ ]:
import inspect
import os
import re
from dataclasses import dataclass
from functools import partial
from typing import Callable
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# lsst.summit.utils imports spectractor (through lsst.atmospec), which calls matplotlib.use("agg"). Imported after
# the magic below (as get_consdb_client does) it silently drops every later figure, so import it first.
try:
    import lsst.summit.utils  # noqa: F401
except ImportError:
    pass
%matplotlib inline
pd.set_option("display.width", 160)
NOTEBOOK_DIR = os.getcwd()
try:
    not_times_square
except NameError:  # Times Square replaced the parameters cell, which defines the flag
    not_times_square = False

# Bumped through the PTH Method Change Process; the baseline commit is tagged rosp-07-vX.Y.
METHOD_VERSION = "rosp-07-v1.0"

SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PLATESCALE = 0.2  # arcsec / pixel, LSSTCam
ARCSEC_TO_RAD = np.pi / (180.0 * 3600.0)
REFERENCE_WAVELENGTH_NM = 500.0
# ConsDB exp_midpt is TAI, the EFD time index is UTC (TAI - UTC = 37 s since 2017; no leap second is scheduled)
TAI_MINUS_UTC = pd.Timedelta(seconds=37)
EXCLUDE_REASONS = ("filter_change_close_loop", "twilight_near_sun")
MIN_VISITS_PER_NIGHT = (
    5  # a monitor's nightly sDIQ needs at least this many valid visits
)
# Effective wavelengths (nm) copied from lsst.summit.utils.getBandpassSeeingCorrection for LSSTCam (as ROSP-2);
# the Provenance cell at the end checks them against the kernel's summit_utils.
# Upstream versions the method is pinned to. The Provenance cell compares them with the summit_utils loaded in this
# kernel and with the Rapid Analysis package versions the ConsDB records for the selected visits; a change in any
# of them is a Method Change (the guider sDIQ depends on the tomographic layer coefficients and the von Karman
# defaults in lsst.summit.utils.guiders.seeing, the PSF and guider columns on the Rapid Analysis producer).
PINNED = {
    "summit_utils": "w.2026.44",
    "guider_seeing": {  # lsst.summit.utils.guiders.seeing at summit_utils w.2026.44
        "sha256": "cc2ba5238dedf8985a81ad0c36f182fa7ea686cb79738cdff43b90dc384560f5",
        "outer_scale_m": 25.0,  # fwhmVonkarman default; the producer (rubintv_production guiders.py) does not override it
        "tomographic_coefficients": {  # CorrelationAnalysis.calcTomographicCoeffs: ((diag_x, diag_y), (cross_x, cross_y), (adj_x, adj_y))
            "low": ((0, 0.51), (0.49, 0), (0, 0)),
            "high": ((0, 0), (0, 0), (0.872, 0.128)),
        },
    },
    "ess_csc_version": None,  # RINGSS ESS CSC (salIndex 304) cscVersion; None = recorded but not pinned yet
}
EFFECTIVE_WAVELENGTH_SOURCE = f"lsst.summit.utils.utils.getBandpassSeeingCorrection, summit_utils {PINNED['summit_utils']}"
RA_TRACKED_PACKAGES = ("summit_utils", "rubintv_production", "lsst_distrib")
EFFECTIVE_WAVELENGTH_NM = {
    "u": 365.1397,
    "g": 475.6104,
    "r": 618.6055,
    "i": 753.5490,
    "z": 868.0935,
    "y": 972.5979,
}
BANDS = ("u", "g", "r", "i", "z", "y")
BAND_COLORS = dict(
    zip(BANDS, ("#4a3aa7", "#2a78d6", "#1baf7a", "#eda100", "#eb6834", "#e34948"))
)
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "legend.frameon": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "savefig.dpi": 150,
    }
)


@dataclass(frozen=True)
class Monitor:
    """One atmospheric seeing estimate: its input column, how that column is brought to the frame of the visit
    (and, for reference, to zenith / 500 nm), and its plot style. Both conversions take (values, airmass, band).
    """

    tag: str
    label: str
    source: str  # column holding the monitor value as the producer reports it
    to_visit_frame: Callable  # -> seeing (arcsec) at the airmass and band of the visit: what is subtracted
    to_zenith_500nm: Callable  # -> reference seeing (arcsec) at zenith / 500 nm: tables and plots only
    conversion: str  # one line on to_visit_frame, for the Provenance cell
    color: str
    marker: str

    @property
    def z500(self):
        return f"{self.tag}_zenith_500nm"

    @property
    def atm(self):
        return f"atm_{self.tag}"

    @property
    def sys2(self):
        return f"sys2_{self.tag}"


def make_monitors(L0, min_wind):
    """RINGSS and the DIMM both report Kolmogorov seeing at zenith / 500 nm and get the same r0 path with the outer
    scale L0 (None = Kolmogorov FWHM); the guider is already in the frame of the visit.
    """
    l0_label = f"L0 = {L0:g} m" if L0 else "no outer-scale correction"
    conversion = (
        f"r0 (lambda_eff/500 nm)^1.2 X^-0.6, then the von Karman FWHM with L0 = {L0:g} m"
        if L0
        else "r0 (lambda_eff/500 nm)^1.2 X^-0.6, then the Kolmogorov FWHM (no outer scale)"
    )
    dimm_label = f"DIMM, {l0_label}"
    if min_wind > 0:
        dimm_label += f", wind >= {min_wind:g} m/s"
    return (
        Monitor(
            tag="ringss",
            label=f"RINGSS scintillation, {l0_label}",
            source="ringss_z500",  # Kolmogorov seeing at zenith / 500 nm
            to_visit_frame=partial(kolmogorov_seeing_in_visit_frame, L0=L0),
            to_zenith_500nm=partial(kolmogorov_seeing_at_zenith_500nm, L0=L0),
            conversion=conversion,
            color="#2a78d6",
            marker="o",
        ),
        Monitor(
            tag="dimm",
            label=dimm_label,
            source="dimm_seeing_windy",  # Kolmogorov seeing at zenith / 500 nm, wind cut applied
            to_visit_frame=partial(kolmogorov_seeing_in_visit_frame, L0=L0),
            to_zenith_500nm=partial(kolmogorov_seeing_at_zenith_500nm, L0=L0),
            conversion=conversion,
            color="#eb6834",
            marker="s",
        ),
        # The guider tomographic seeing (lsst.summit.utils.guiders.seeing) is measured along the line of sight and
        # scaled to the exposure wavelength by its producer, so it is already in the frame of the visit.
        Monitor(
            tag="guider",
            label="guider mid + high layers",
            source="guider_free_atm",
            to_visit_frame=as_is,
            to_zenith_500nm=observed_to_zenith_500nm,
            conversion="none, used as stored (already at the airmass and wavelength of the visit)",
            color="#1baf7a",
            marker="^",
        ),
    )


def parse_day_obs(value) -> int:
    s = str(value).strip().replace("-", "")
    if not re.fullmatch(r"\d{8}", s):
        raise ValueError(f"Not a day_obs: {value!r}")
    return int(s)


def parse_programs(value) -> list:
    """Comma/space separated block names; bare numbers become BLOCK-<n>; empty means no program cut."""
    items = [v for v in re.split(r"[,\s]+", str(value or "").strip()) if v]
    return [f"BLOCK-{it}" if re.fullmatch(r"T?\d+", it) else it for it in items]


def parse_bool(value) -> bool:
    return str(value).strip().lower() in ("true", "1", "yes")


def parse_outer_scale(value):
    """Outer scale in metres, or None for no correction (0, empty, 'none', 'inf')."""
    s = str(value).strip().lower()
    if s in ("", "none", "inf", "nan", "0", "0.0"):
        return None
    L0 = float(s)
    return L0 if L0 > 0 else None


def day_obs_chunks(day_obs_min, day_obs_max, chunk_days=31):
    lo, end = pd.Timestamp(str(day_obs_min)), pd.Timestamp(str(day_obs_max))
    while lo <= end:
        hi = min(lo + pd.Timedelta(days=chunk_days - 1), end)
        yield int(lo.strftime("%Y%m%d")), int(hi.strftime("%Y%m%d"))
        lo = hi + pd.Timedelta(days=1)


def day_obs_window(day_obs_lo, day_obs_hi):
    """UTC start/end (astropy Time, as lsst_efd_client requires) covering the nights day_obs_lo..day_obs_hi:
    12:00 UTC to 12:00 UTC the day after."""
    from astropy.time import Time

    t0 = datetime.strptime(str(day_obs_lo), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    )
    t1 = datetime.strptime(str(day_obs_hi), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    ) + pd.Timedelta(days=1)
    return Time(t0), Time(t1)


def _read_cache(cache_path, use_cache):
    if cache_path and use_cache and os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache {cache_path}")
        return df
    return None


def _write_cache(cache_path, df):
    if cache_path and len(df):
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        df.to_parquet(cache_path, index=False)
    return df


def cached(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Run loader() unless cache_path already holds its parquet output."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, loader())


async def cached_async(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Same as cached() for a loader that returns a coroutine (the EFD client is async on the RSP)."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, await loader())


def _visit_airmass_and_wavelength(airmass, band):
    """(X, lambda_eff in nm) as float arrays for a visit: the one place the airmass is coerced and the band
    (first character, stripped) is mapped to its effective wavelength; NaN where either is unknown.
    """
    X = pd.to_numeric(pd.Series(airmass), errors="coerce").to_numpy(dtype=float)
    wavelength = (
        pd.Series(band)
        .astype("string")
        .str.strip()
        .str[0]
        .map(EFFECTIVE_WAVELENGTH_NM)
        .astype(float)
        .to_numpy()
    )
    return X, wavelength


def zenith_500nm_factor(airmass, band):
    """Multiply an observed FWHM by this to refer it to zenith and 500 nm: X^-0.6 (lambda_eff / 500 nm)^0.2."""
    X, wavelength = _visit_airmass_and_wavelength(airmass, band)
    with np.errstate(divide="ignore", invalid="ignore"):
        return (
            np.where(X >= 1.0, X**-0.6, np.nan)
            * (wavelength / REFERENCE_WAVELENGTH_NM) ** 0.2
        )


def fried_parameter(seeing_arcsec, wavelength_nm=REFERENCE_WAVELENGTH_NM):
    """r0 (m) from a Kolmogorov FWHM in arcsec: r0 = 0.98 lambda / FWHM."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return (
            0.98
            * wavelength_nm
            * 1e-9
            / (np.asarray(seeing_arcsec, dtype=float) * ARCSEC_TO_RAD)
        )


def outer_scale_factor(r0_m, L0):
    """sqrt(1 - 2.18 (r0/L0)^0.356) (Tokovinin 2002); 1 for L0=None; NaN where the bracket is not positive."""
    r0 = np.asarray(r0_m, dtype=float)
    if L0 is None:
        return np.where(np.isfinite(r0), 1.0, np.nan)
    with np.errstate(invalid="ignore"):
        bracket = 1.0 - 2.18 * np.power(r0 / float(L0), 0.356)
        return np.where(bracket > 0, np.sqrt(np.clip(bracket, 0, None)), np.nan)


def von_karman_seeing(r0_m, wavelength_nm, L0):
    """FWHM (arcsec) at wavelength_nm for a Fried parameter r0_m (m) at that wavelength and outer scale L0."""
    r0 = np.asarray(r0_m, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        kolmogorov = (
            0.98 * np.asarray(wavelength_nm, dtype=float) * 1e-9 / r0 / ARCSEC_TO_RAD
        )
    return kolmogorov * outer_scale_factor(r0, L0)


def kolmogorov_seeing_in_visit_frame(seeing, airmass, band, L0):
    """Kolmogorov seeing at zenith / 500 nm (RINGSS, DIMM) -> von Karman FWHM at the visit airmass and band, r0
    scaled first: r0(lambda, X) = r0 (lambda_eff / 500 nm)^1.2 X^-0.6. With L0 = None this is the Kolmogorov
    X^0.6 (lambda_eff / 500 nm)^-0.2 de-correction (the inverse of ROSP-2)."""
    X, wavelength = _visit_airmass_and_wavelength(airmass, band)
    with np.errstate(divide="ignore", invalid="ignore"):
        r0 = (
            fried_parameter(seeing)
            * (wavelength / REFERENCE_WAVELENGTH_NM) ** 1.2
            * X**-0.6
        )
    return np.where(X >= 1.0, von_karman_seeing(r0, wavelength, L0), np.nan)


def kolmogorov_seeing_at_zenith_500nm(seeing, airmass, band, L0):
    """Reference only: Kolmogorov seeing at zenith / 500 nm as the von Karman FWHM there, no de-correction."""
    return von_karman_seeing(fried_parameter(seeing), REFERENCE_WAVELENGTH_NM, L0)


def as_is(values, airmass, band):
    """The monitor value unchanged (already in the target frame)."""
    return np.asarray(values, dtype=float)


def observed_to_zenith_500nm(values, airmass, band):
    """Seeing in the frame of the visit -> zenith / 500 nm reference: X^-0.6 (lambda_eff / 500 nm)^0.2."""
    return np.asarray(values, dtype=float) * zenith_500nm_factor(airmass, band)


def signed_sqrt(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.sqrt(np.abs(x))

## ConsDB access

`visit1` (pointing, band, wind, DIMM) joined to `visit1_quicklook` (PSF, guider seeing) and `exposure`
(`observation_reason`), one row per science visit, chunked by calendar month, through
`lsst.summit.utils.ConsDbClient` from the kernel image (the in-cluster USDF endpoint, or `LSST_CONSDB_PQ_URL` when set).
`fetch_visit_efd` reads the per-visit RINGSS mean from the transformed-EFD table `efd_lsstcam.visit1_efd`,
the first-choice RINGSS source for every visit (next section). `fetch_rapid_analysis_versions` reads the package
versions Rapid Analysis records per exposure (`exposure_quicklook.ra_package_versions`, written by the same pods
that produce the PSF and guider columns) for the Provenance cell.

In [ ]:
USDF_CONSDB_URL = (
    "http://consdb-pq.consdb:8080/consdb"  # in-cluster (Times Square, USDF Nublado)
)
VISIT_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "band",
    "physical_filter",
    "science_program",
    "img_type",
    "exp_time",
    "exp_midpt",
    "airmass",
    "zenith_distance",
    "altitude",
    "azimuth",
    "wind_speed",
    "wind_dir",
    "dimm_seeing",
)
QUICKLOOK_COLUMNS = (
    "psf_sigma_median",
    "n_psf_star_total",
    "guider_mid_layer_seeing",
    "guider_free_seeing",
)
EXPOSURE_COLUMNS = ("observation_reason",)
VISIT_EFD_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "mt_ringss_fwhm_scintillation",
)
TEXT_COLUMNS = (
    "band",
    "physical_filter",
    "science_program",
    "observation_reason",
    "img_type",
    "exp_midpt",
    "created_at",
)


USDF_CONSDB_URL_EXTERNAL = (
    "https://usdf-rsp.slac.stanford.edu/consdb"  # outside the cluster (SDF nodes)
)
RSP_TOKEN_FILE = os.path.expanduser(
    "~/.lsst/usdf_rsp"
)  # Gafaelfawr token (gt-...) for the external endpoint


def _consdb_endpoint(url=None):
    """(url, token): explicit url or LSST_CONSDB_PQ_URL, else in-cluster, else the external USDF endpoint.

    Outside the cluster (e.g. an SDF interactive node) consdb-pq.consdb does not resolve, so use the
    public USDF RSP endpoint with a Gafaelfawr token from ACCESS_TOKEN or RSP_TOKEN_FILE.
    """
    import socket

    url = url or os.environ.get("LSST_CONSDB_PQ_URL")
    if url:
        return url, None
    try:
        socket.getaddrinfo("consdb-pq.consdb", 8080)
        return USDF_CONSDB_URL, None
    except socket.gaierror:
        pass
    token = os.environ.get("ACCESS_TOKEN")
    if not token and os.path.exists(RSP_TOKEN_FILE):
        with open(RSP_TOKEN_FILE) as f:
            token = f.read().strip()
    if not token:
        raise RuntimeError(
            "Not in the USDF cluster and no RSP token: set ACCESS_TOKEN or put a gt-... token in "
            f"{RSP_TOKEN_FILE} (or set LSST_CONSDB_PQ_URL)."
        )
    return USDF_CONSDB_URL_EXTERNAL, token


class ConsDb:
    """lsst.summit.utils.ConsDbClient with .query(sql) -> DataFrame."""

    def __init__(self, url=None):
        from lsst.summit.utils import ConsDbClient

        self.url, token = _consdb_endpoint(url)
        os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
        self._client = ConsDbClient(self.url, token=token)

    def query(self, sql):
        result = self._client.query(sql)
        return result if isinstance(result, pd.DataFrame) else result.to_pandas()


def get_consdb_client():
    client = ConsDb()
    print(f"ConsDB client: lsst.summit.utils.ConsDbClient at {client.url}")
    return client


def _sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def _numeric(df):
    for c in df.columns:
        if c not in TEXT_COLUMNS and df[c].dtype == object:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def build_query(
    day_obs_min, day_obs_max, programs, instrument="lsstcam", img_types=("science",)
):
    schema = f"cdb_{instrument}"
    where = [f"v.day_obs >= {int(day_obs_min)}", f"v.day_obs <= {int(day_obs_max)}"]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return (
        "SELECT\n    "
        + ",\n    ".join(
            [f"v.{c}" for c in VISIT_COLUMNS]
            + [f"q.{c}" for c in QUICKLOOK_COLUMNS]
            + [f"e.{c}" for c in EXPOSURE_COLUMNS]
        )
        + f"\nFROM {schema}.visit1 AS v"
        + f"\nLEFT JOIN {schema}.visit1_quicklook AS q ON q.visit_id = v.visit_id"
        + f"\nLEFT JOIN {schema}.exposure AS e ON e.exposure_id = v.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY v.day_obs, v.seq_num"
    )


def fetch_visits(client, day_obs_min, day_obs_max, programs, chunk_days=31):
    cols = list(VISIT_COLUMNS) + list(QUICKLOOK_COLUMNS) + list(EXPOSURE_COLUMNS)
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_query(lo, hi, programs))
        if not set(cols).issubset(df.columns):
            raise RuntimeError(
                f"ConsDB returned an invalid schema for {lo}-{hi}; refusing partial results"
            )
        print(f"  consdb {lo}-{hi}: {len(df):6d} visits")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=cols)
    out = _numeric(pd.concat(frames, ignore_index=True))
    return out.sort_values(["day_obs", "seq_num"]).reset_index(drop=True)


def fetch_visit_efd(
    client, day_obs_min, day_obs_max, instrument="lsstcam", chunk_days=31
):
    """Per-visit RINGSS scintillation mean from efd_<instrument>.visit1_efd."""
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(
            "SELECT "
            + ", ".join(VISIT_EFD_COLUMNS)
            + f" FROM efd_{instrument}.visit1_efd"
            + f" WHERE day_obs >= {int(lo)} AND day_obs <= {int(hi)}"
        )
        print(f"  visit1_efd {lo}-{hi}: {len(df):6d} rows")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=list(VISIT_EFD_COLUMNS))
    out = _numeric(pd.concat(frames, ignore_index=True))
    return out.sort_values(["day_obs", "seq_num"]).reset_index(drop=True)


def fetch_rapid_analysis_versions(
    client, day_obs_min, day_obs_max, instrument="lsstcam", chunk_days=31
):
    """exposure_id, day_obs, seq_num and the ra_package_versions JSON Rapid Analysis recorded per exposure.

    The column is JSONB ({"hash": ..., "versions": {package: version}}) and may arrive as a string or a dict;
    it is parsed into a dict column `versions` (None where empty).
    """
    import json

    schema = f"cdb_{instrument}"
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(
            "SELECT q.exposure_id, e.day_obs, e.seq_num, q.ra_package_versions"
            f" FROM {schema}.exposure_quicklook AS q"
            f" JOIN {schema}.exposure AS e ON e.exposure_id = q.exposure_id"
            f" WHERE e.day_obs >= {int(lo)} AND e.day_obs <= {int(hi)}"
            " AND q.ra_package_versions IS NOT NULL"
        )
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["exposure_id", "day_obs", "seq_num", "versions"])
    out = pd.concat(frames, ignore_index=True)

    def parse(value):
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return None
        if isinstance(value, (str, bytes)):
            value = json.loads(value)
        return dict(value.get("versions", {})) or None

    out["versions"] = out.pop("ra_package_versions").map(parse)
    out["exposure_id"] = pd.to_numeric(out["exposure_id"], errors="coerce")
    return out[["exposure_id", "day_obs", "seq_num", "versions"]]


def fetch_guider_l0(client, day_obs_min, day_obs_max, instrument="lsstcam"):
    """Distinct non-null visit1_quicklook.guider_l0 values over the range (the producer currently writes none)."""
    df = client.query(
        f"SELECT DISTINCT guider_l0 FROM cdb_{instrument}.visit1_quicklook"
        f" WHERE day_obs >= {int(day_obs_min)} AND day_obs <= {int(day_obs_max)} AND guider_l0 IS NOT NULL"
    )
    return pd.to_numeric(df["guider_l0"], errors="coerce").dropna().tolist()

## EFD access: RINGSS

RINGSS reports at zenith / 500 nm. Two sources are pulled over the night range, independently of each other: the per-visit
mean already in the ConsDB (`efd_lsstcam.visit1_efd.mt_ringss_fwhm_scintillation`, `fetch_visit_efd` above) and the raw
`logevent_ringssMeasurement` stream (salIndex 304) from the EFD through `lsst_efd_client` (`fetch_ringss`). `attach_ringss`
then gives each visit `ringss_consdb` (the ConsDB value) and `ringss_efd` (the median of the EFD measurements within
± `ringss_window_minutes` of the exposure midpoint, which is TAI in the ConsDB and is converted to UTC for the EFD;
NaN when the window holds none), and picks `ringss_z500` per visit: the ConsDB value when it is finite and positive,
else the EFD median when there is one, else NaN. `ringss_src` records the choice (`consdb`, `efd` or `none`) for every
visit, so an EFD outage or retention gap on some nights only costs the visits that have neither. A pull that fails or
returns nothing is reported with a warning and in the Provenance cell. `ringss_source` can restrict the pull to one source.


In [ ]:
RINGSS_TOPIC = "lsst.sal.ESS.logevent_ringssMeasurement"
RINGSS_SAL_INDEX = 304
RINGSS_FIELD = "fwhmScintillation"
RINGSS_SOURCES = (
    "both",
    "consdb",
    "efd",
)  # both: visit1_efd first, EFD window where it is missing


def get_efd_client(efd_name="usdf_efd"):
    import lsst_efd_client

    return lsst_efd_client.EfdClient(efd_name, output_mode="dataframe")


async def fetch_ringss(efd_client, day_obs_min, day_obs_max, chunk_days=31):
    """All RINGSS scintillation measurements from 12:00 UTC on day_obs_min to 12:00 UTC after day_obs_max."""
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        start, end = day_obs_window(lo, hi)
        df = efd_client.select_time_series(
            RINGSS_TOPIC, [RINGSS_FIELD], start, end, index=RINGSS_SAL_INDEX
        )
        if inspect.isawaitable(df):
            df = await df
        print(f"  ringss {lo}-{hi}: {len(df):6d} measurements")
        if len(df):
            df = df.reset_index().rename(columns={df.index.name or "index": "time"})
            frames.append(df[["time", RINGSS_FIELD]])
    if not frames:
        return pd.DataFrame(columns=["time", RINGSS_FIELD])
    out = pd.concat(frames, ignore_index=True)
    out["time"] = pd.to_datetime(out["time"], utc=True).dt.tz_localize(None)
    out[RINGSS_FIELD] = pd.to_numeric(out[RINGSS_FIELD], errors="coerce")
    return out.dropna().sort_values("time").reset_index(drop=True)


async def load_ringss(
    source, consdb, efd_name, day_obs_min, day_obs_max, cache_dir, use_cache
):
    """Pull the RINGSS sources selected by `source` ("both", "consdb" or "efd").

    Returns {"consdb": visit1_efd rows or None, "efd": EFD measurements or None, "notes": [...]}: a pull that
    fails or returns nothing leaves None and a note (printed as a warning, repeated in the Provenance cell).
    """
    tables = {"consdb": None, "efd": None, "notes": []}

    def note(msg):
        print(f"WARNING: {msg}")
        tables["notes"].append(msg)

    if source in ("both", "consdb"):
        try:
            df = cached(
                cache_dir
                and os.path.join(
                    cache_dir, f"visit1_efd_{day_obs_min}_{day_obs_max}.parquet"
                ),
                lambda: fetch_visit_efd(consdb, day_obs_min, day_obs_max),
                use_cache,
            )
            if len(df):
                tables["consdb"] = df
            else:
                note("ConsDB visit1_efd has no RINGSS rows for this range")
        except Exception as exc:
            note(f"RINGSS from ConsDB visit1_efd failed ({exc!r})")
    if source in ("both", "efd"):
        try:
            efd = get_efd_client(efd_name)
            df = await cached_async(
                cache_dir
                and os.path.join(
                    cache_dir, f"ringss_efd_{day_obs_min}_{day_obs_max}.parquet"
                ),
                lambda: fetch_ringss(efd, day_obs_min, day_obs_max),
                use_cache,
            )
            if len(df):
                tables["efd"] = df
            else:
                note("the EFD returned no RINGSS measurements for this range")
        except Exception as exc:
            note(f"RINGSS pull from the EFD failed ({exc!r})")
    return tables


def efd_window_median(visits, table, window_minutes):
    """Median of the EFD RINGSS measurements within +/- window_minutes of each visit midpoint (NaN if none)."""
    t_r = table["time"].to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    vals = table[RINGSS_FIELD].to_numpy(dtype=float)
    t_v = (
        pd.to_datetime(visits["exp_midpt"], utc=True, errors="coerce", format="ISO8601")
        - TAI_MINUS_UTC
    ).dt.tz_localize(None).to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    w = window_minutes * 60.0
    lo = np.searchsorted(t_r, t_v - w, side="left")
    hi = np.searchsorted(t_r, t_v + w, side="right")
    med = np.full(len(visits), np.nan)
    for i, (a, b) in enumerate(zip(lo, hi)):
        if b > a:
            med[i] = np.median(vals[a:b])
    return med


def attach_ringss(visits, tables, window_minutes):
    """Add the RINGSS columns to a copy of visits.

    ringss_consdb: the visit1_efd per-visit mean; ringss_efd: median of the EFD measurements within +/- window_minutes
    of the visit midpoint (NaN if none); ringss_z500 (zenith / 500 nm, what the sDIQ uses): ringss_consdb where it is
    finite and positive, else ringss_efd where finite; ringss_src: "consdb", "efd" or "none".
    """
    out = visits.copy()
    out["ringss_consdb"] = np.nan
    out["ringss_efd"] = np.nan
    if tables["consdb"] is not None:
        efd = tables["consdb"].drop_duplicates("visit_id")[
            ["visit_id", "mt_ringss_fwhm_scintillation"]
        ]
        values = pd.to_numeric(
            efd.set_index("visit_id")["mt_ringss_fwhm_scintillation"], errors="coerce"
        )
        out["ringss_consdb"] = out["visit_id"].map(values).astype(float)
    if tables["efd"] is not None:
        out["ringss_efd"] = efd_window_median(out, tables["efd"], window_minutes)
    use_consdb = np.isfinite(out["ringss_consdb"]) & (out["ringss_consdb"] > 0)
    use_efd = ~use_consdb & np.isfinite(out["ringss_efd"])
    out["ringss_z500"] = np.where(
        use_consdb, out["ringss_consdb"], np.where(use_efd, out["ringss_efd"], np.nan)
    )
    out["ringss_src"] = np.select([use_consdb, use_efd], ["consdb", "efd"], "none")
    return out


ESS_VERSIONS_TOPIC = "lsst.sal.ESS.logevent_softwareVersions"
ESS_VERSION_FIELDS = ["cscVersion", "xmlVersion", "salVersion"]


async def fetch_ess_versions(efd_client, day_obs_min, day_obs_max, lookback_days=365):
    """softwareVersions events of the RINGSS ESS CSC (salIndex 304) from lookback_days before the range to its end.

    The event is logged when the CSC starts, so the version in effect at the start of the range is the last event
    before it; the lookback finds that one.
    """
    from astropy.time import Time

    start, _ = day_obs_window(day_obs_min, day_obs_min)
    _, end = day_obs_window(day_obs_max, day_obs_max)
    df = efd_client.select_time_series(
        ESS_VERSIONS_TOPIC,
        ESS_VERSION_FIELDS,
        Time(start.utc.datetime - timedelta(days=lookback_days)),
        end,
        index=RINGSS_SAL_INDEX,
    )
    if inspect.isawaitable(df):
        df = await df
    if not len(df):
        return pd.DataFrame(columns=["time", *ESS_VERSION_FIELDS])
    df = df.reset_index().rename(columns={df.index.name or "index": "time"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    return df[["time", *ESS_VERSION_FIELDS]].sort_values("time").reset_index(drop=True)

## Data, FWHM, atmosphere and selection

The ConsDB pull, the RINGSS attachment, the per-visit FWHM and the three atmospheric estimates in the frame of
the visit (`atm_<monitor>`, each monitor's own `to_visit_frame` applied to its source column: the r0 path with the
outer scale for RINGSS and the DIMM, identity for the guider), with `<monitor>_zenith_500nm` alongside
for reference (`to_zenith_500nm`; for RINGSS and the DIMM the outer-scale-corrected value at zenith / 500 nm), then
`sys2_<monitor>` = FWHM² − atm².
The DIMM value is blanked on visits below `dimm_min_wind_speed` or without a wind reading. The selection table is the
ROSP-2 one; the counts after it are the visits each monitor can use.

In [ ]:
start_dayobs, end_dayobs = parse_day_obs(start_dayobs), parse_day_obs(end_dayobs)
if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )
programs = parse_programs(science_programs)
min_psf_stars = int(min_psf_stars)
if min_psf_stars < 0:
    raise ValueError(f"min_psf_stars ({min_psf_stars}) must be >= 0")
outer_scale_m = parse_outer_scale(outer_scale_m)
dimm_min_wind_speed = float(dimm_min_wind_speed or 0)
ringss_source = str(ringss_source).strip().lower()
if ringss_source not in RINGSS_SOURCES:
    raise ValueError(
        f"ringss_source must be one of {RINGSS_SOURCES}, not {ringss_source!r}"
    )
ringss_window_minutes = float(ringss_window_minutes)
use_cache = parse_bool(use_cache)
MONITORS = make_monitors(outer_scale_m, dimm_min_wind_speed)
cache_dir = (
    os.path.join(NOTEBOOK_DIR, "cache") if (use_cache and not_times_square) else None
)
cache_tag = f"v2_{start_dayobs}_{end_dayobs}_{'_'.join(p.replace('BLOCK-', '') for p in programs) or 'all'}"
print(
    f"dayObs {start_dayobs} to {end_dayobs}; programs {programs or 'all'}; "
    f"min_psf_stars {min_psf_stars}; outer scale L0 {outer_scale_m}; DIMM wind >= {dimm_min_wind_speed:g} m/s; "
    f"RINGSS source {ringss_source}"
)

consdb = get_consdb_client()
raw = cached(
    cache_dir and os.path.join(cache_dir, f"visits_{cache_tag}.parquet"),
    lambda: fetch_visits(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)
ringss_tables = await load_ringss(
    ringss_source, consdb, efd_name, start_dayobs, end_dayobs, cache_dir, use_cache
)
visits = attach_ringss(raw, ringss_tables, ringss_window_minutes)

visits["band"] = visits["band"].astype("string").str.strip().str[0]
visits["fwhm"] = visits["psf_sigma_median"].astype(float) * SIGMA_TO_FWHM * PLATESCALE
visits["time"] = pd.to_datetime(
    visits["exp_midpt"], utc=True, errors="coerce", format="ISO8601"
).dt.tz_localize(None)
dimm = visits["dimm_seeing"].astype(float)
if (
    dimm_min_wind_speed > 0
):  # the DIMM is unreliable in low wind: drop it there (and where the wind is unknown)
    windy = (visits["wind_speed"].astype(float) >= dimm_min_wind_speed).fillna(False)
    dimm = dimm.where(windy)
visits["dimm_seeing_windy"] = dimm
visits["guider_free_atm"] = np.sqrt(
    visits["guider_mid_layer_seeing"].astype(float) ** 2
    + visits["guider_free_seeing"].astype(float) ** 2
)
# each monitor brings its own value to the frame of the visit (what is subtracted) and to zenith / 500 nm (reference)
for m in MONITORS:
    value = visits[m.source].astype(float)
    visits[m.atm] = m.to_visit_frame(value, visits["airmass"], visits["band"])
    visits[m.z500] = m.to_zenith_500nm(value, visits["airmass"], visits["band"])
    visits[m.sys2] = visits["fwhm"] ** 2 - visits[m.atm] ** 2

steps = [
    ("has PSF FWHM", visits["fwhm"].notna()),
    (
        f"n_psf_star_total >= {min_psf_stars}",
        visits["n_psf_star_total"].astype(float) >= min_psf_stars,
    ),
    (
        "has airmass >= 1 and a known band",
        (visits["airmass"] >= 1) & visits["band"].isin(BANDS),
    ),
    (
        f"observation_reason not in {EXCLUDE_REASONS}",
        ~visits["observation_reason"].isin(EXCLUDE_REASONS),
    ),
]
keep = pd.Series(True, index=visits.index)
rows = []
for name, mask in steps:
    mask = mask.fillna(False).astype(bool)
    rows.append((name, int((keep & ~mask).sum()), int((keep & mask).sum())))
    keep &= mask
print(f"\nStart: {len(visits)} visits")
print(
    pd.DataFrame(rows, columns=["cut", "removed", "remaining"]).to_string(index=False)
)
sel = visits[keep].copy()
for m in MONITORS:
    valid = np.isfinite(sel[m.atm]) & (sel[m.atm] > 0)
    sel.loc[~valid, [m.z500, m.atm, m.sys2]] = np.nan
print(
    f"\n{len(sel)} visits on {sel['day_obs'].nunique()} nights; with a usable monitor value:"
)
print(
    pd.DataFrame(
        {
            m.label: {
                "visits": int(sel[m.sys2].notna().sum()),
                "fraction": sel[m.sys2].notna().mean(),
                "nights": int(sel.loc[sel[m.sys2].notna(), "day_obs"].nunique()),
            }
            for m in MONITORS
        }
    ).T.round(3)
)
ringss_monitor = next(m for m in MONITORS if m.tag == "ringss")
# per-visit RINGSS source among the selected visits (none = neither source had a value)
sel["ringss_src"] = sel["ringss_src"].where(sel[ringss_monitor.sys2].notna(), "none")
ringss_src_table = pd.DataFrame(
    {
        src: {
            "visits": int((sel["ringss_src"] == src).sum()),
            "nights": int(sel.loc[sel["ringss_src"] == src, "day_obs"].nunique()),
        }
        for src in ("consdb", "efd", "none")
    }
).T
print(
    "\nRINGSS source per selected visit (consdb = visit1_efd mean, efd = EFD window median):"
)
print(ringss_src_table)
efd_only_nights = sorted(
    set(sel.loc[sel["ringss_src"] == "efd", "day_obs"])
    - set(sel.loc[sel["ringss_src"] == "consdb", "day_obs"])
)
if efd_only_nights:
    print(
        f"nights whose RINGSS comes from the EFD only: {len(efd_only_nights)} "
        f"({efd_only_nights[0]} .. {efd_only_nights[-1]})"
    )
for m in MONITORS:
    if not sel[m.sys2].notna().any():
        print(
            f"WARNING: no usable {m.label} values for dayObs {start_dayobs}-{end_dayobs}; its sDIQ is NaN."
        )
print()
print(
    pd.crosstab(sel["science_program"], sel["band"], margins=True).reindex(
        columns=[*BANDS, "All"], fill_value=0
    )
)

In [ ]:
# Sanity check on the de-correction: the factor that takes a zenith / 500 nm monitor reading to the frame of the
# visit (atm in the frame of the visit / reading; RINGSS and the DIMM share it), for each band at a few airmasses.
# The Kolmogorov factor X^0.6 (lambda_eff / 500 nm)^-0.2 is the same for any reading. With an outer scale the
# factor depends on the reading (a larger r0 is reduced more), so the applied one is shown for three readings.
probe = pd.DataFrame(
    {"airmass": np.repeat([1.0, 1.2, 1.5, 2.0], len(BANDS)), "band": list(BANDS) * 4}
)


def factor_table(factor):
    """One factor per probe row -> airmass x band table."""
    table = probe.assign(factor=factor).pivot(
        index="airmass", columns="band", values="factor"
    )
    return table[list(BANDS)]


kolmogorov_label = "Kolmogorov, any reading" + ("" if outer_scale_m else " (applied)")
factors = {
    kolmogorov_label: factor_table(
        1.0 / zenith_500nm_factor(probe["airmass"], probe["band"])
    )
}
if outer_scale_m:
    for reading in (0.6, 0.8, 1.2):  # monitor reading, arcsec at zenith / 500 nm
        atm = ringss_monitor.to_visit_frame(
            np.full(len(probe), reading), probe["airmass"], probe["band"]
        )
        factors[f'applied: L0 = {outer_scale_m:g} m, reading {reading:g}"'] = (
            factor_table(atm / reading)
        )
pd.concat(factors, names=["factor", "airmass"]).round(3)

## Results

### Aggregate table

One row per monitor: the sDIQ (arcsec) with its 25-75 % range (`sDIQ p25`, `sDIQ p75`: the signed square roots of the 25th
and 75th percentiles of the per-visit $\mathrm{sys}^2$, i.e. the spread of the visits, not an uncertainty on the sDIQ),
the number of visits and nights behind it, the medians of
the ingredients (delivered FWHM, the monitor seeing in the frame of the visit and at zenith / 500 nm), and the
median, 25th and 75th percentiles of the per-visit $\mathrm{sys}^2$ (arcsec²) with the fraction of visits where it is negative.

In [ ]:
def sdiq_stats(sys2):
    """Median and quartiles of sys2 (arcsec^2) and their signed square roots: the sDIQ and its 25-75 % range.

    The range is the spread of the visits around the sDIQ, not the uncertainty of the sDIQ.
    """
    s = np.asarray(sys2, dtype=float)
    s = s[np.isfinite(s)]
    keys = (
        "n",
        "sdiq",
        "sdiq_p25",
        "sdiq_p75",
        "median_sys2",
        "p25_sys2",
        "p75_sys2",
        "frac_negative",
    )
    if not len(s):
        return dict.fromkeys(keys, np.nan) | {"n": 0}
    med, p25, p75 = np.median(s), np.percentile(s, 25), np.percentile(s, 75)
    return dict(
        n=len(s),
        sdiq=float(signed_sqrt(med)),
        sdiq_p25=float(signed_sqrt(p25)),
        sdiq_p75=float(signed_sqrt(p75)),
        median_sys2=med,
        p25_sys2=p25,
        p75_sys2=p75,
        frac_negative=float((s < 0).mean()),
    )


def aggregate_table(df, monitors):
    rows = []
    for m in monitors:
        ok = df[m.sys2].notna()
        st = sdiq_stats(df.loc[ok, m.sys2])
        rows.append(
            {
                "monitor": m.label,
                "sDIQ": st["sdiq"],
                "sDIQ p25": st["sdiq_p25"],
                "sDIQ p75": st["sdiq_p75"],
                "n_visits": st["n"],
                "n_nights": int(df.loc[ok, "day_obs"].nunique()),
                "median FWHM": df.loc[ok, "fwhm"].median(),
                "median atm (visit frame)": df.loc[ok, m.atm].median(),
                "median atm (zenith/500nm)": df.loc[ok, m.z500].median(),
                "median airmass": df.loc[ok, "airmass"].median(),
                "sys2 p25": st["p25_sys2"],
                "sys2 median": st["median_sys2"],
                "sys2 p75": st["p75_sys2"],
                "frac sys2 < 0": st["frac_negative"],
            }
        )
    return pd.DataFrame(rows).set_index("monitor")


summary = aggregate_table(sel, MONITORS)
print(
    "sDIQ [25 %, 75 %]: signed square roots of the median and of the 25th / 75th percentiles of the per-visit sys2"
)
for m in MONITORS:
    row = summary.loc[m.label]
    print(
        f'{m.label}: sDIQ = {row["sDIQ"]:.3f} [{row["sDIQ p25"]:.3f}, {row["sDIQ p75"]:.3f}] arcsec  '
        f'(median sys2 {row["sys2 median"]:+.4f} arcsec2; n = {int(row["n_visits"])} visits on {int(row["n_nights"])} nights)'
    )
summary.round(3).T

### Per band

sDIQ per monitor for each band (and all bands), with the same 25-75 % range (`p25`, `p75`) and the number of visits.

The sDIQ is quoted at the wavelength of each band. A system term that is itself turbulence (dome or mirror seeing) scales
with wavelength like the atmosphere, FWHM $\propto \lambda^{-0.2}$, so it is expected to rise toward the blue. `sDIQ 500nm` removes
that scaling: it is the same statistic on the per-visit $\mathrm{sys}^2\,(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{0.4}$, which within a band
is sDIQ $\times\,(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{0.2}$. It is flat across the bands if the band dependence of the sDIQ is that of
turbulence, and its all-bands row then does not depend on the band mix. It is a diagnostic, not the metric. A band dependence
that remains in `sDIQ 500nm` points at the de-correction of the monitors or at band-specific system terms; the figure
"sDIQ against wavelength" in the sanity checks shows the same comparison for the 25 %, 50 % and 75 % values.

In [ ]:
def sys2_at_500nm(df, monitor):
    """Per-visit sys2 referred to 500 nm with the turbulence scaling.

    A seeing-like (Kolmogorov) term has FWHM ~ lambda^-0.2, so sys2 ~ lambda^-0.4 and
    sys2 x (lambda_eff / 500 nm)^0.4 is its value at 500 nm. A diagnostic of the band dependence, not the metric.
    """
    _, wavelength = _visit_airmass_and_wavelength(df["airmass"], df["band"])
    return df[monitor.sys2] * (wavelength / REFERENCE_WAVELENGTH_NM) ** 0.4


def band_table(df, monitors):
    groups = [("all", df)] + [
        (b, df[df["band"] == b]) for b in BANDS if (df["band"] == b).any()
    ]
    rows = []
    for name, g in groups:
        row = {"band": name, "median airmass": g["airmass"].median()}
        for m in monitors:
            st = sdiq_stats(g[m.sys2])
            row[f"sDIQ {m.tag}"] = st["sdiq"]
            row[f"p25 {m.tag}"] = st["sdiq_p25"]
            row[f"p75 {m.tag}"] = st["sdiq_p75"]
            row[f"sDIQ 500nm {m.tag}"] = sdiq_stats(sys2_at_500nm(g, m))["sdiq"]
            row[f"n {m.tag}"] = st["n"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("band")


band_table(sel, MONITORS).round(3)

### Nightly time series

Top: the nightly median delivered FWHM and each monitor's seeing in the frame of the visit, with 25-75 % whiskers
over the visits of the night. Bottom: the nightly sDIQ per monitor, the square root of that night's median
$\mathrm{sys}^2$, with whiskers at the signed square roots of the night's 25th and 75th percentiles of the per-visit
$\mathrm{sys}^2$. Nights with fewer than 5 usable visits for a monitor are left out for that monitor; series are offset slightly in x.

In [ ]:
def nightly_table(df, monitors, min_visits=MIN_VISITS_PER_NIGHT):
    def q(x, p):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return np.percentile(x, p) if len(x) else np.nan

    rows = []
    for day_obs, sub in df.groupby("day_obs"):
        row = dict(
            day_obs=int(day_obs),
            date=pd.Timestamp(str(int(day_obs))),
            n_visits=len(sub),
            fwhm=q(sub["fwhm"], 50),
            fwhm_p25=q(sub["fwhm"], 25),
            fwhm_p75=q(sub["fwhm"], 75),
        )
        for m in monitors:
            st = sdiq_stats(sub[m.sys2])
            ok = st["n"] >= min_visits
            row[f"n_{m.tag}"] = st["n"]
            row[m.atm] = q(sub[m.atm], 50) if ok else np.nan
            row[f"{m.atm}_p25"] = q(sub[m.atm], 25) if ok else np.nan
            row[f"{m.atm}_p75"] = q(sub[m.atm], 75) if ok else np.nan
            row[f"sdiq_{m.tag}"] = st["sdiq"] if ok else np.nan
            row[f"sdiq_{m.tag}_p25"] = (
                float(signed_sqrt(st["p25_sys2"])) if ok else np.nan
            )
            row[f"sdiq_{m.tag}_p75"] = (
                float(signed_sqrt(st["p75_sys2"])) if ok else np.nan
            )
        rows.append(row)
    if not rows:  # no selected visits: keep the columns the plots expect
        cols = ["day_obs", "date", "n_visits", "fwhm", "fwhm_p25", "fwhm_p75"]
        for m in monitors:
            cols += [f"n_{m.tag}", m.atm, f"{m.atm}_p25", f"{m.atm}_p75"]
            cols += [f"sdiq_{m.tag}", f"sdiq_{m.tag}_p25", f"sdiq_{m.tag}_p75"]
        return pd.DataFrame(columns=cols)
    return pd.DataFrame(rows)


def whiskers(ax, nightly, col, color, marker, label, offset_days=0.0, ms=4):
    x = nightly["date"] + pd.Timedelta(days=offset_days)
    ax.errorbar(
        x,
        nightly[col],
        yerr=np.array(
            [nightly[col] - nightly[f"{col}_p25"], nightly[f"{col}_p75"] - nightly[col]]
        ),
        fmt=marker,
        ms=ms,
        color=color,
        ecolor=color,
        elinewidth=1.0,
        capsize=0,
        alpha=0.9,
        label=label,
    )


nightly = nightly_table(sel, MONITORS)
offsets = np.linspace(-0.2, 0.2, len(MONITORS) + 1)
fig, axes = plt.subplots(
    2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [1, 1.3]}
)
ax = axes[0]
whiskers(ax, nightly, "fwhm", INK, "o", "delivered PSF FWHM", offsets[0])
for k, m in enumerate(MONITORS):
    whiskers(
        ax,
        nightly,
        m.atm,
        m.color,
        m.marker,
        f"{m.label} (visit frame)",
        offsets[k + 1],
    )
ax.set_ylabel("per-visit value (arcsec)")
ax.set_title(
    f"{len(sel)} visits on {sel['day_obs'].nunique()} nights", loc="left", fontsize=11
)
ax.legend(
    ncol=2,
    fontsize=9,
    title="marker: nightly median; whisker: 25-75 %",
    title_fontsize=9,
)
ax = axes[1]
for k, m in enumerate(MONITORS):
    whiskers(ax, nightly, f"sdiq_{m.tag}", m.color, m.marker, m.label, offsets[k + 1])
ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
ax.set_ylabel("sDIQ = sqrt(median FWHM² - atm²) (arcsec)")
ax.set_xlabel("night (day_obs)")
ax.legend(
    ncol=3,
    fontsize=9,
    title="marker: nightly sDIQ; whisker: signed sqrt of the 25-75 % range of per-visit FWHM² - atm²",
    title_fontsize=9,
)
fig.align_ylabels(axes)
plt.show()

nightly.set_index("day_obs").drop(columns="date").round(3)

### Histograms

Per-visit $\mathrm{sys}^2$ = FWHM² − atm² for each monitor, as a fraction of that monitor's visits per bin, with the
median (solid) and 25 / 75 % (dotted) of each marked above the curves. The legend quotes the sDIQ (the square root of the median) and the three values.

In [ ]:
bins = np.arange(-1.0, 2.0 + 0.025, 0.025)
fig, ax = plt.subplots(figsize=(11, 6.5))
ymax = 0
stats = {}
for m in MONITORS:
    d = sel[m.sys2].dropna()
    if not len(d):
        continue
    st = stats[m.tag] = sdiq_stats(d)
    counts, _, _ = ax.hist(
        d.clip(bins[0], bins[-1]),
        bins=bins,
        histtype="step",
        lw=2,
        color=m.color,
        weights=np.full(len(d), 1 / len(d)),
        label=(
            f'{m.label}: sDIQ {st["sdiq"]:.3f}"; sys² {st["median_sys2"]:+.3f} '
            f'[{st["p25_sys2"]:+.3f}, {st["p75_sys2"]:+.3f}] arcsec²  (n = {st["n"]})'
        ),
    )
    ymax = max(ymax, counts.max())
for k, m in enumerate(MONITORS):
    if m.tag not in stats:
        continue
    st = stats[m.tag]
    y = ymax * (1.06 + 0.06 * k)
    ax.plot([st["p25_sys2"], st["p75_sys2"]], [y, y], color=m.color, lw=1.2, ls=":")
    ax.plot([st["median_sys2"]], [y], marker="|", ms=12, mew=2, color=m.color)
    ax.axvline(st["median_sys2"], color=m.color, lw=1.0, ls="-", alpha=0.6)
ax.axvline(0, color=INK_MUTED, lw=1, ls="--")
ax.set_ylim(0, ymax * (1.06 + 0.06 * len(MONITORS) + 0.04))
ax.set_xlim(bins[0], bins[-1])
ax.set_xlabel("PSF FWHM² - atmosphere² (arcsec²)")
ax.set_ylabel("fraction of visits per bin")
ax.set_title(f"{start_dayobs}-{end_dayobs}, {', '.join(programs) or 'all programs'}")
ax.legend(
    fontsize=9,
    title="sDIQ; median [25 %, 75 %] of sys²",
    title_fontsize=9,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.13),
)
plt.show()

### Sanity checks

**sDIQ against elevation.** One panel per monitor: the sDIQ (square root of the binned median of $\mathrm{sys}^2$) in bins of telescope altitude, for all bands (black) and per band,
with the visits per bin underneath. The whiskers are the 25-75 % range quoted with the sDIQ in the tables, here over the visits of the bin
(the signed square roots of the 25th and 75th percentiles of the per-visit $\mathrm{sys}^2$, as in the nightly series). The all-band point sits at the
centre of its bin and the bands are drawn next to it, on both sides, so that the whiskers do not hide each other: every point
between two bin limits (those of the histogram underneath) belongs to that bin.
The atmosphere is removed in the frame of the visit, so a residual trend with elevation would point at the $X^{0.6}$
scaling of a monitor or at a real elevation dependence of the system (mirror support, wind exposure). Bins with fewer than 10 visits are skipped.

In [ ]:
MIN_VISITS_PER_BIN = 10


def binned_sdiq(x, sys2, edges, min_count=MIN_VISITS_PER_BIN):
    """sdiq_stats of sys2 in bins of x; bins with fewer than min_count visits are left NaN."""
    x, s = np.asarray(x, dtype=float), np.asarray(sys2, dtype=float)
    ok = np.isfinite(x) & np.isfinite(s)
    idx = np.digitize(x[ok], edges) - 1
    rows = []
    for i in range(len(edges) - 1):
        st = sdiq_stats(s[ok][idx == i])
        if st["n"] < min_count:
            st = dict.fromkeys(st, np.nan) | {"n": st["n"]}
        rows.append(
            {
                "center": 0.5 * (edges[i] + edges[i + 1]),
                "width": edges[i + 1] - edges[i],
                **st,
            }
        )
    return pd.DataFrame(rows)


def bin_shifts(n_series, span=0.7):
    """Sideways offsets, in bin widths, for n_series drawn in one bin.

    The first series stays at the bin centre and the others are spread on both sides of it, never on top of it.
    """
    n_side = n_series - 1
    slots = np.linspace(-0.5 * span, 0.5 * span, n_side + 1 + n_side % 2)
    return [0.0, *np.delete(slots, len(slots) // 2)[:n_side]]


def plot_binned_sdiq(ax, st, color, label, main=False, shift=0.0):
    """One series of binned_sdiq(): the sDIQ of each bin joined by a line, with whiskers over the visits of the bin.

    The whisker is the 25-75 % range of sdiq_stats (signed square roots of the 25th and 75th percentiles of the
    per-visit sys2 in the bin, as quoted in the tables and drawn in the nightly series): the spread of the visits,
    not the uncertainty of the sDIQ. shift (in bin widths, see bin_shifts) moves the series sideways so that the
    whiskers of several series do not hide each other. The whiskers of the series that are not the main one are
    drawn lighter than their markers.
    """
    series = ax.errorbar(
        st["center"] + shift * st["width"],
        st["sdiq"],
        yerr=np.array([st["sdiq"] - st["sdiq_p25"], st["sdiq_p75"] - st["sdiq"]]),
        marker="o" if main else "s",
        ms=5 if main else 3.5,
        lw=2 if main else 1.2,
        elinewidth=1.5 if main else 1.0,
        capsize=0,
        color=color,
        label=label,
        zorder=3 if main else 2.5,
    )
    if not main:
        for whisker in series[2]:
            whisker.set_alpha(0.5)


def legend_above(fig, axes_row):
    """One legend for the figure, above the panels, with every series drawn in any of them."""
    entries = {}
    for ax in axes_row:
        for handle, name in zip(*ax.get_legend_handles_labels()):
            entries.setdefault(name, handle)
    if entries:
        fig.legend(
            entries.values(),
            entries.keys(),
            loc="outside upper center",
            ncol=len(entries),
            fontsize=9,
            title="marker: sDIQ of the bin; whisker: signed sqrt of the 25-75 % range of per-visit FWHM² - atm²",
            title_fontsize=9,
        )


alt_edges = np.arange(20.0, 90.1, 5.0)
groups = [("all", INK, sel)] + [
    (b, BAND_COLORS[b], sel[sel["band"] == b])
    for b in BANDS
    if (sel["band"] == b).any()
]
fig, axes = plt.subplots(
    2,
    len(MONITORS),
    figsize=(5.6 * len(MONITORS), 8),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
    squeeze=False,
    layout="constrained",
)
for k, m in enumerate(MONITORS):
    ax, ax_n = axes[0, k], axes[1, k]
    for (name, color, g), shift in zip(groups, bin_shifts(len(groups))):
        st = binned_sdiq(g["altitude"], g[m.sys2], alt_edges)
        if st["sdiq"].notna().any():
            series = "all bands" if name == "all" else f"{name} band"
            plot_binned_sdiq(ax, st, color, series, main=name == "all", shift=shift)
        d = g.loc[g[m.sys2].notna(), "altitude"].dropna()
        if len(d):
            ax_n.hist(
                d,
                bins=alt_edges,
                histtype="step",
                lw=2 if name == "all" else 1.2,
                color=color,
            )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_title(f"{m.label}  (n = {int(sel[m.sys2].notna().sum())})", fontsize=11)
    ax.set_ylabel("sDIQ = sqrt(binned median FWHM² - atm²) (arcsec)")
    ax_n.set_xlabel("telescope elevation (deg)")
    ax_n.set_ylabel("visits")
    ax_n.set_xlim(alt_edges[0], alt_edges[-1])
legend_above(fig, axes[0])
fig.align_ylabels(axes[:, 0])
plt.show()

**sDIQ against the delivered FWHM.** The same binned sDIQ with the same 25-75 % whiskers, now in bins of the as-delivered PSF FWHM of the visit, all bands and per band,
with the visits per bin underneath. A system term that is
independent of the seeing gives a flat curve; a steady rise with the FWHM points at a term that scales with the seeing
(a monitor that under-reads by a fixed fraction, dome or mirror seeing that tracks the free atmosphere). Some rise is
built in, because binning on the FWHM also bins on the noise of $\mathrm{sys}^2$: at a given atmosphere the visits with the
largest FWHM are those with positive fluctuations. Bins with fewer than 10 visits are skipped.

In [ ]:
lo = np.floor(sel["fwhm"].quantile(0.005) * 10) / 10
hi = np.ceil(sel["fwhm"].quantile(0.995) * 10) / 10
fwhm_edges = np.arange(lo, hi + 0.05, 0.1)
fig, axes = plt.subplots(
    2,
    len(MONITORS),
    figsize=(5.6 * len(MONITORS), 8),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
    squeeze=False,
    layout="constrained",
)
for k, m in enumerate(MONITORS):
    ax, ax_n = axes[0, k], axes[1, k]
    for (name, color, g), shift in zip(groups, bin_shifts(len(groups))):
        st = binned_sdiq(g["fwhm"], g[m.sys2], fwhm_edges)
        if st["sdiq"].notna().any():
            series = "all bands" if name == "all" else f"{name} band"
            plot_binned_sdiq(ax, st, color, series, main=name == "all", shift=shift)
        d = g.loc[g[m.sys2].notna(), "fwhm"].dropna()
        if len(d):
            ax_n.hist(
                d,
                bins=fwhm_edges,
                histtype="step",
                lw=2 if name == "all" else 1.2,
                color=color,
            )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_title(f"{m.label}  (n = {int(sel[m.sys2].notna().sum())})", fontsize=11)
    ax.set_ylabel("sDIQ = sqrt(binned median FWHM² - atm²) (arcsec)")
    ax_n.set_xlabel("PSF FWHM as delivered (arcsec)")
    ax_n.set_ylabel("visits")
    ax_n.set_xlim(fwhm_edges[0], fwhm_edges[-1])
legend_above(fig, axes[0])
fig.align_ylabels(axes[:, 0])
plt.show()

**sDIQ against the monitor seeing.** The same binned sDIQ with the same 25-75 % whiskers, now in bins of each monitor's own atmospheric seeing in the frame of
the visit (`atm_<monitor>`, the quantity that is subtracted), all bands and per band, with the visits per bin underneath. This is the
complement of the previous plot, with the bias reversed: the monitor value does not carry the noise of the PSF measurement but
carries its own (measurement error, and a line of sight and averaging time that differ from the telescope's), so visits land in
a high bin partly because the monitor over-read, and even a system term that is independent of the atmosphere falls with
increasing monitor seeing here. Beyond that, a fall means the monitor over-reads by a fraction of the seeing (or $L_0$ is too
large), a rise means it under-reads (or the outer scale is too small), since a fractional error $f$ in atm enters $\mathrm{sys}^2$
as $\mp 2 f\,\mathrm{atm}^2$. Bins with fewer than 10 visits are skipped.

In [ ]:
fig, axes = plt.subplots(
    2,
    len(MONITORS),
    figsize=(5.6 * len(MONITORS), 8),
    gridspec_kw={"height_ratios": [3, 1]},
    squeeze=False,
    layout="constrained",
)
for k, m in enumerate(MONITORS):
    ax, ax_n = axes[0, k], axes[1, k]
    atm = sel.loc[sel[m.sys2].notna(), m.atm]
    if atm.empty:
        ax.set_title(f"{m.label}  (n = 0)", fontsize=11)
        continue
    lo = np.floor(atm.quantile(0.005) * 10) / 10
    hi = np.ceil(atm.quantile(0.995) * 10) / 10
    atm_edges = np.arange(lo, hi + 0.05, 0.1)
    for (name, color, g), shift in zip(groups, bin_shifts(len(groups))):
        st = binned_sdiq(g[m.atm], g[m.sys2], atm_edges)
        if st["sdiq"].notna().any():
            series = "all bands" if name == "all" else f"{name} band"
            plot_binned_sdiq(ax, st, color, series, main=name == "all", shift=shift)
        d = g.loc[g[m.sys2].notna(), m.atm].dropna()
        if len(d):
            ax_n.hist(
                d,
                bins=atm_edges,
                histtype="step",
                lw=2 if name == "all" else 1.2,
                color=color,
            )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_title(f"{m.label}  (n = {len(atm)})", fontsize=11)
    ax.set_ylabel("sDIQ = sqrt(binned median FWHM² - atm²) (arcsec)")
    ax.tick_params(labelbottom=False)
    ax_n.set_xlabel(f"{m.tag} seeing at the visit, atm_{m.tag} (arcsec)")
    ax_n.set_ylabel("visits")
    for a in (ax, ax_n):
        a.set_xlim(atm_edges[0], atm_edges[-1])
legend_above(fig, axes[0])
fig.align_ylabels(axes[:, 0])
plt.show()

**sDIQ against wind speed.** The same binned sDIQ with the same 25-75 % whiskers, now in 2 m/s bins of the outside wind speed of the
visit (`visit1.wind_speed`), all bands and per band, with the visits per bin underneath. A system term that falls as the wind rises points
at dome or mirror seeing that the wind flushes; one that rises with the wind points at wind shake. The monitors can depend on the wind too:
the DIMM is unreliable in low wind and is only used at or above `dimm_min_wind_speed`, so its panel starts there. Bins with fewer than 10
visits are skipped.

In [ ]:
def plot_sdiq_in_bins(column, edges, xlabel, series):
    """The sanity-check figure for one per-visit quantity: per monitor, the binned sDIQ of each series with its
    25-75 % whiskers (top) and the visits per bin (bottom).

    series: (label, colour, visits) per curve, the main one (drawn at the bin centre) first.
    """
    fig, axes = plt.subplots(
        2,
        len(MONITORS),
        figsize=(5.6 * len(MONITORS), 8),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
        squeeze=False,
        layout="constrained",
    )
    for k, m in enumerate(MONITORS):
        ax, ax_n = axes[0, k], axes[1, k]
        shifts = bin_shifts(len(series))
        for i, ((label, color, g), shift) in enumerate(zip(series, shifts)):
            st = binned_sdiq(g[column], g[m.sys2], edges)
            if st["sdiq"].notna().any():
                plot_binned_sdiq(ax, st, color, label, main=i == 0, shift=shift)
            d = g.loc[g[m.sys2].notna(), column].dropna()
            if len(d):
                ax_n.hist(
                    d,
                    bins=edges,
                    histtype="step",
                    lw=2 if i == 0 else 1.2,
                    color=color,
                )
        n = int((sel[m.sys2].notna() & sel[column].notna()).sum())
        ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
        ax.set_title(f"{m.label}  (n = {n})", fontsize=11)
        ax.set_ylabel("sDIQ = sqrt(binned median FWHM² - atm²) (arcsec)")
        ax_n.set_xlabel(xlabel)
        ax_n.set_ylabel("visits")
        ax_n.set_xlim(edges[0], edges[-1])
    legend_above(fig, axes[0])
    fig.align_ylabels(axes[:, 0])
    return fig, axes


WIND_BIN = 2.0  # m/s
top = sel["wind_speed"].quantile(0.995)
top = WIND_BIN * np.ceil(top / WIND_BIN) if np.isfinite(top) else 16.0
wind_edges = np.arange(0.0, top + 0.5 * WIND_BIN, WIND_BIN)
band_series = [
    ("all bands" if name == "all" else f"{name} band", color, g)
    for name, color, g in groups
]
plot_sdiq_in_bins("wind_speed", wind_edges, "outside wind speed (m/s)", band_series)
plt.show()

**sDIQ against the pointing relative to the wind.** The same binned sDIQ and whiskers in 30° bins of the angle between the telescope
azimuth and the direction the wind comes from (`visit1.azimuth` − `visit1.wind_dir`, folded to 0-180°: 0° = pointing into the wind,
90° = wind from the side, 180° = wind from behind; the two sides of the wind are taken together). All bands together: black for all
wind speeds, and blue from light to dark for the visits below 4 m/s, at 4 to 8 m/s and at 8 m/s or more, with the visits per bin
underneath. An effect of the wind should grow from the light to the dark series. A dependence on the angle that is the same at every wind
speed, or that is there in calm air (where the wind direction is poorly defined), is more likely something else that varies with azimuth.
`wind_dir` is taken as the direction the wind blows from (the meteorological convention); if it were the direction it blows to, 0° and
180° would be exchanged. Bins with fewer than 10 visits are skipped.

In [ ]:
WIND_SPEED_CLASSES = (0.0, 4.0, 8.0, np.inf)  # m/s, edges of the wind-speed split
WIND_CLASS_COLORS = ("#86b6ef", "#3987e5", "#1c5cab")  # one hue, light to dark

# 0 = pointing into the wind, 180 = wind from behind; the two sides of the wind are folded together
sel["wind_angle"] = np.abs((sel["azimuth"] - sel["wind_dir"] + 180.0) % 360.0 - 180.0)
angle_edges = np.arange(0.0, 180.1, 30.0)
wind_series = [("all wind speeds", INK, sel)]
for lo, hi, color in zip(
    WIND_SPEED_CLASSES[:-1], WIND_SPEED_CLASSES[1:], WIND_CLASS_COLORS
):
    if lo == 0:
        label = f"wind < {hi:g} m/s"
    elif np.isinf(hi):
        label = f"wind >= {lo:g} m/s"
    else:
        label = f"wind {lo:g} to {hi:g} m/s"
    in_class = (sel["wind_speed"] >= lo) & (sel["wind_speed"] < hi)
    wind_series.append((label, color, sel[in_class]))
fig, axes = plot_sdiq_in_bins(
    "wind_angle",
    angle_edges,
    "pointing relative to the wind (deg)\n0 = into the wind, 180 = wind from behind",
    wind_series,
)
axes[1, 0].set_xticks(angle_edges)
plt.show()

**sDIQ against wavelength.** One panel per monitor, each band at its effective wavelength: the marker is the sDIQ of the band and
the whisker its 25-75 % range (the signed square roots of the 50th, 25th and 75th percentiles of the per-visit $\mathrm{sys}^2$ of the band,
as in the per-band table). The black lines are the turbulence model. A system term that is itself turbulence (dome or mirror seeing)
scales with wavelength like the atmosphere, FWHM $\propto \lambda^{-0.2}$, so the model is $A\,(\lambda/500\,\mathrm{nm})^{-0.2}$ with no free slope.
There is one line per percentile (solid: 50 %, dotted: 25 % and 75 %), and its amplitude $A$ is that percentile of the per-visit
$\mathrm{sys}^2\,(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{0.4}$ over all bands together (signed square root); for the solid line this is the all-bands
`sDIQ 500nm` of the per-band table. Top: as measured, where the model is a curve. Bottom: the same values multiplied by
$(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{0.2}$, i.e. referred to 500 nm, where the model is a horizontal line at $A$ (quoted in the panel title).

If the markers and the whisker ends follow the lines, the band dependence is that of turbulence. A slope that is left in the bottom row is a
wavelength dependence the model does not explain: falling toward the red, the sDIQ is steeper than $\lambda^{-0.2}$; rising toward the red, it is
shallower (an achromatic term rises as $\lambda^{0.2}$ there). The amplitudes come from all bands together, so they are set mostly by the bands
with the most visits (the number of visits is given under each band). The 25 % and 75 % values also carry the scatter of the monitor and of the
PSF measurement, which need not scale this way. Where a value is negative (the monitor reads more than the delivered FWHM) the model has no
physical meaning: the line only shows the same scaling applied to a negative $\mathrm{sys}^2$. Bands with fewer than 10 visits are skipped.

In [ ]:
from matplotlib.lines import Line2D

TURBULENCE_EXPONENT = -0.2  # FWHM ~ lambda^-0.2 for a turbulent (Kolmogorov) term
wavelength_grid = np.linspace(330.0, 1010.0, 200)
# (sdiq_stats key, line style, line width): the 50 % value is the sDIQ
MODEL_LINES = (("sdiq", "-", 2.0), ("sdiq_p25", ":", 1.6), ("sdiq_p75", ":", 1.6))

fig, axes = plt.subplots(
    2,
    len(MONITORS),
    figsize=(5.6 * len(MONITORS), 8.5),
    sharex="col",
    squeeze=False,
    layout="constrained",
)
for k, m in enumerate(MONITORS):
    ax, ax_500 = axes[0, k], axes[1, k]
    ax.set_title(f"{m.label}  (n = {int(sel[m.sys2].notna().sum())})", fontsize=11)
    ax.set_ylabel("sDIQ at the wavelength of the band (arcsec)")
    ax_500.set_ylabel(
        r"sDIQ $\times\ (\lambda_\mathrm{eff}\,/\,500\ \mathrm{nm})^{0.2}$ (arcsec)"
    )
    ax_500.set_xlabel("band, effective wavelength and visits")
    ax_500.set_xlim(wavelength_grid[0], wavelength_grid[-1])
    # model amplitudes: percentiles of the per-visit sys2 referred to 500 nm, all bands together
    model = sdiq_stats(sys2_at_500nm(sel, m))
    if not model["n"]:
        continue
    for key, ls, lw in MODEL_LINES:
        scaling = (wavelength_grid / REFERENCE_WAVELENGTH_NM) ** TURBULENCE_EXPONENT
        ax.plot(wavelength_grid, model[key] * scaling, color=INK, ls=ls, lw=lw)
        ax_500.axhline(model[key], color=INK, ls=ls, lw=lw)
    ticks = {}
    for b in BANDS:
        st = sdiq_stats(sel.loc[sel["band"] == b, m.sys2])
        if st["n"] < MIN_VISITS_PER_BIN:
            continue
        wavelength = EFFECTIVE_WAVELENGTH_NM[b]
        to_500nm = (wavelength / REFERENCE_WAVELENGTH_NM) ** -TURBULENCE_EXPONENT
        for a, factor in ((ax, 1.0), (ax_500, to_500nm)):
            a.errorbar(
                wavelength,
                st["sdiq"] * factor,
                yerr=[
                    [(st["sdiq"] - st["sdiq_p25"]) * factor],
                    [(st["sdiq_p75"] - st["sdiq"]) * factor],
                ],
                marker="o",
                ms=8,
                mec=SURFACE,
                mew=1.0,
                color=BAND_COLORS[b],
                elinewidth=1.6,
                capsize=5,
                capthick=1.6,
                zorder=3,
            )
        ticks[wavelength] = f"{b}\n{wavelength:.0f} nm\nn = {st['n']}"
    ax_500.set_title(
        f'model at 500 nm: A = {model["sdiq"]:.3f} [{model["sdiq_p25"]:.3f}, {model["sdiq_p75"]:.3f}] arcsec',
        fontsize=10,
    )
    if ticks:
        ax_500.set_xticks(list(ticks), labels=list(ticks.values()), fontsize=8)
    for a in (ax, ax_500):
        if a.get_ylim()[0] < 0 < a.get_ylim()[1]:
            a.axhline(0, color=INK_MUTED, lw=1, ls="--")
fig.legend(
    handles=[
        Line2D([], [], color=INK_MUTED, marker="o", ms=8, lw=1.6),
        Line2D([], [], color=INK, ls="-", lw=2.0),
        Line2D([], [], color=INK, ls=":", lw=1.6),
    ],
    labels=[
        "band (colour as in the other figures): sDIQ, whisker to the 25 % and 75 % values",
        r"turbulence model for the sDIQ: $A\,(\lambda\,/\,500\ \mathrm{nm})^{-0.2}$, $A$ from all bands",
        "the same for the 25 % and 75 % values",
    ],
    loc="outside upper center",
    ncol=3,
    fontsize=9,
)
fig.align_ylabels(axes[:, 0])
plt.show()

**Histograms per band.** The per-visit $\mathrm{sys}^2$ of each monitor split by band (fraction of that band's visits per bin), with the
median (tick) and 25 / 75 % (dotted) of each band marked above the curves and the all-band median as the vertical line. The legend
quotes the per-band sDIQ with its 25-75 % range. A band whose distribution sits apart from the others has either a chromatic system term or a monitor scaling problem.

In [ ]:
fig, axes = plt.subplots(
    1, len(MONITORS), figsize=(6 * len(MONITORS), 7.5), sharey=True, squeeze=False
)
for ax, m in zip(axes[0], MONITORS):
    ymax, stats = 0, {}
    for b in BANDS:
        d = sel.loc[sel["band"] == b, m.sys2].dropna()
        if len(d) < 2:
            continue
        st = stats[b] = sdiq_stats(d)
        counts, _, _ = ax.hist(
            d.clip(bins[0], bins[-1]),
            bins=bins,
            histtype="step",
            lw=1.8,
            color=BAND_COLORS[b],
            weights=np.full(len(d), 1 / len(d)),
            label=(
                f'{b}: sDIQ {st["sdiq"]:.3f} [{st["sdiq_p25"]:.3f}, {st["sdiq_p75"]:.3f}]"  '
                f'(n = {st["n"]})'
            ),
        )
        ymax = max(ymax, counts.max())
    for k, (b, st) in enumerate(stats.items()):
        y = ymax * (1.06 + 0.05 * k)
        ax.plot(
            [st["p25_sys2"], st["p75_sys2"]],
            [y, y],
            color=BAND_COLORS[b],
            lw=1.2,
            ls=":",
        )
        ax.plot(
            [st["median_sys2"]], [y], marker="|", ms=10, mew=2, color=BAND_COLORS[b]
        )
    st_all = sdiq_stats(sel[m.sys2])
    ax.axvline(
        st_all["median_sys2"],
        color=INK,
        lw=1.2,
        label=f'all bands: sDIQ {st_all["sdiq"]:.3f} [{st_all["sdiq_p25"]:.3f}, {st_all["sdiq_p75"]:.3f}]"',
    )
    ax.axvline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_ylim(0, ymax * (1.06 + 0.05 * len(stats) + 0.04) if ymax else 1)
    ax.set_xlim(bins[0], bins[-1])
    ax.set_xlabel("PSF FWHM² - atmosphere² (arcsec²)")
    ax.set_title(m.label, fontsize=11)
    ax.legend(
        fontsize=8.5,
        title="sDIQ [25 %, 75 %]; ticks above the curves: median [25 %, 75 %] of sys²",
        title_fontsize=8.5,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
    )
axes[0, 0].set_ylabel("fraction of the band's visits per bin")
plt.show()

## Provenance

What this run was computed with: the method version, the inputs (PSF column, the three monitors and the RINGSS
sources used per visit, the outer scale), the kernel image and the library versions, and the **upstream pins**
(`PINNED` in the setup cell) checked three ways:

* the `lsst.summit.utils` loaded in this kernel: its version, the effective wavelengths (compared with
  `getBandpassSeeingCorrection` for the physical filters in the selection) and the guider seeing module
  (`guiders/seeing.py` SHA-256, the `fwhmVonkarman` outer-scale default and the tomographic layer coefficients);
* the Rapid Analysis package versions the ConsDB records per exposure (`exposure_quicklook.ra_package_versions`:
  `summit_utils`, `rubintv_production`, `lsst_distrib`) for the selected visits, i.e. what actually produced
  `psf_sigma_median` and the `guider_*` columns, with the `summit_utils` entry compared with the pin, plus the
  `visit1_quicklook.guider_l0` column (the producer does not write it, so the module default applies);
* the RINGSS ESS CSC (salIndex 304) `softwareVersions` events from the EFD, in effect over the range. The RINGSS
  reduction code itself and the `visit1_efd` transform are not versioned anywhere queryable, which is noted.

A `MISMATCH` or `DIFFERS` in any of these means an upstream input changed, and the Method Change Process decides
whether the notebook follows.


In [ ]:
import hashlib
import platform
import shutil
import subprocess


def kernel_summit_utils_version():
    try:
        from lsst.summit.utils import version as su_version

        return str(su_version.__version__)
    except Exception:
        pass
    try:
        from importlib.metadata import version

        return version("lsst-summit-utils")
    except Exception:
        return None


def check_effective_wavelengths(df):
    """Compare EFFECTIVE_WAVELENGTH_NM with the summit_utils of this kernel, per physical filter in df."""
    try:
        from lsst.summit.utils.utils import getBandpassSeeingCorrection
    except ImportError:
        return "not checked (lsst.summit.utils not importable in this kernel)"
    problems = []
    pairs = (
        df[["band", "physical_filter"]]
        .dropna()
        .drop_duplicates()
        .itertuples(index=False)
    )
    for band, physical_filter in pairs:
        try:
            installed_nm = (
                500.0 * getBandpassSeeingCorrection(str(physical_filter)) ** 5
            )
        except ValueError as exc:
            problems.append(f"{physical_filter}: {exc}")
            continue
        if abs(installed_nm - EFFECTIVE_WAVELENGTH_NM[band]) > 0.01:
            problems.append(
                f"{physical_filter}: notebook {EFFECTIVE_WAVELENGTH_NM[band]:.4f} nm, "
                f"installed {installed_nm:.4f} nm"
            )
    return "match" if not problems else "MISMATCH - " + "; ".join(problems)


def check_guider_seeing_module():
    """Compare the kernel's lsst.summit.utils.guiders.seeing with the pinned hash, outer scale and coefficients."""
    try:
        from lsst.summit.utils.guiders import seeing
    except ImportError:
        return "not checked (lsst.summit.utils.guiders.seeing not importable in this kernel)"
    pin = PINNED["guider_seeing"]
    parts = []
    try:
        with open(inspect.getsourcefile(seeing), "rb") as f:
            digest = hashlib.sha256(f.read()).hexdigest()
        parts.append(
            f"seeing.py sha256 {digest[:12]} "
            + (
                "= pin"
                if digest == pin["sha256"]
                else f"MISMATCH (pin {pin['sha256'][:12]})"
            )
        )
    except Exception as exc:
        parts.append(f"seeing.py not hashable ({exc!r})")
    try:
        L0 = inspect.signature(seeing.fwhmVonkarman).parameters["outerScale"].default
        parts.append(
            f"L0 default {L0:g} m "
            + (
                "= pin"
                if L0 == pin["outer_scale_m"]
                else f"MISMATCH (pin {pin['outer_scale_m']:g} m)"
            )
        )
    except Exception as exc:
        parts.append(f"L0 default not readable ({exc!r})")
    try:
        coeffs = seeing.CorrelationAnalysis.calcTomographicCoeffs(None)
        parts.append(
            "layer coefficients "
            + (
                "= pin"
                if coeffs == pin["tomographic_coefficients"]
                else f"MISMATCH ({coeffs})"
            )
        )
    except Exception as exc:
        parts.append(f"layer coefficients not readable ({exc!r})")
    return "; ".join(parts)


def summarize_producer_versions(df, versions, packages=RA_TRACKED_PACKAGES):
    """Per tracked package, the versions Rapid Analysis recorded for the visits in df, and the summit_utils pin check."""
    recorded = versions.set_index("exposure_id")["versions"].to_dict()
    per_visit = df["visit_id"].map(recorded)
    have = per_visit.notna()
    lines = []
    for package in packages:
        vals = per_visit[have].map(lambda d: d.get(package, "unknown"))
        if not len(vals):
            continue
        groups = (
            df.loc[have, "day_obs"].groupby(vals.to_numpy()).agg(["min", "max", "size"])
        )
        lines.append(
            f"{package}: "
            + ", ".join(
                f"{v} ({int(g['size'])} visits, {int(g['min'])}..{int(g['max'])})"
                for v, g in groups.sort_values("min").iterrows()
            )
        )
    lines.append(f"{int((~have).sum())} of {len(df)} visits without a record")
    su = set(per_visit[have].map(lambda d: d.get("summit_utils", "unknown")))
    if not su:
        status = "summit_utils pin not checked (no records)"
    elif su == {PINNED["summit_utils"]}:
        status = f"summit_utils = pin {PINNED['summit_utils']}"
    else:
        status = f"summit_utils DIFFERS from pin {PINNED['summit_utils']}: {sorted(su)}"
    return "; ".join(lines), status


def summarize_ess_versions(events, day_obs_min):
    """The ESS CSC software versions in effect over the range: the last event before it and any event within."""
    if not len(events):
        return "no softwareVersions event found (EFD retention?)", "not checked"
    start, _ = day_obs_window(day_obs_min, day_obs_min)
    start = pd.Timestamp(start.utc.datetime)
    before = events[events["time"] < start]
    in_effect = pd.concat([before.tail(1), events[events["time"] >= start]])
    text = "; ".join(
        f"cscVersion {r.cscVersion}, xml {r.xmlVersion}, sal {r.salVersion} (from {r.time:%Y-%m-%d %H:%M} UTC)"
        for r in in_effect.itertuples()
    )
    csc = set(in_effect["cscVersion"].astype(str))
    pin = PINNED["ess_csc_version"]
    if pin is None:
        status = f"recorded, not pinned: {sorted(csc)}"
    elif csc == {pin}:
        status = f"= pin {pin}"
    else:
        status = f"DIFFERS from pin {pin}: {sorted(csc)}"
    return text, status


def stack_version():
    out = []
    v = kernel_summit_utils_version()
    out.append(f"summit_utils {v}" if v else "summit_utils not importable")
    if shutil.which("eups"):
        res = subprocess.run(
            ["eups", "list", "-s", "lsst_distrib"], capture_output=True, text=True
        )
        out.append(f"lsst_distrib {' '.join(res.stdout.split()) or 'not set up'}")
    return "; ".join(out)


# --- the versions behind the data: Rapid Analysis (PSF and guider columns) from the ConsDB, ESS CSC from the EFD
try:
    ra_versions = fetch_rapid_analysis_versions(consdb, start_dayobs, end_dayobs)
    ra_text, ra_status = summarize_producer_versions(sel, ra_versions)
    guider_text, guider_status = summarize_producer_versions(
        sel[sel["guider_free_atm"].notna()], ra_versions
    )
except Exception as exc:
    ra_text = guider_text = f"not available ({exc!r})"
    ra_status = guider_status = "not checked"
try:
    l0_values = fetch_guider_l0(consdb, start_dayobs, end_dayobs)
    l0_text = (
        f"visit1_quicklook.guider_l0 is null for the range: fwhmVonkarman default {PINNED['guider_seeing']['outer_scale_m']:g} m applies"
        if not l0_values
        else f"visit1_quicklook.guider_l0 has values {sorted(set(l0_values))} - CHECK, the pin assumes the module default"
    )
except Exception as exc:
    l0_text = f"visit1_quicklook.guider_l0 not readable ({exc!r})"
try:
    ess_events = await fetch_ess_versions(
        get_efd_client(efd_name), start_dayobs, end_dayobs
    )
    ess_text, ess_status = summarize_ess_versions(ess_events, start_dayobs)
except Exception as exc:
    ess_text, ess_status = f"not available ({exc!r})", "not checked"

kernel_su = kernel_summit_utils_version()
RINGSS_SOURCE_TEXT = {
    "consdb": "ConsDB efd_lsstcam.visit1_efd.mt_ringss_fwhm_scintillation (per-visit mean)",
    "efd": f"EFD {RINGSS_TOPIC} salIndex {RINGSS_SAL_INDEX} {RINGSS_FIELD}, median within +/- {ringss_window_minutes:g} min "
    f"of the visit midpoint ({efd_name})",
}
ringss_order = ("consdb", "efd") if ringss_source == "both" else (ringss_source,)
ringss_description = (
    ("per visit, in priority order: " if len(ringss_order) > 1 else "")
    + "; then ".join(RINGSS_SOURCE_TEXT[s] for s in ringss_order)
    + "; used: "
    + ", ".join(
        f"{int(ringss_src_table.loc[s, 'visits'])} visits from {s}"
        for s in ringss_order
    )
    + f", {int(ringss_src_table.loc['none', 'visits'])} of {len(sel)} selected visits without RINGSS"
    + ("; " + "; ".join(ringss_tables["notes"]) if ringss_tables["notes"] else "")
)
pin_rows = {
    "pinned: summit_utils": PINNED["summit_utils"]
    + " (effective wavelengths; guider seeing module: seeing.py sha256 "
    + PINNED["guider_seeing"]["sha256"][:12]
    + f", L0 {PINNED['guider_seeing']['outer_scale_m']:g} m, layer coefficients {PINNED['guider_seeing']['tomographic_coefficients']})",
    "kernel: summit_utils version": (
        "not importable"
        if not kernel_su
        else (
            f"{kernel_su} = pin"
            if kernel_su == PINNED["summit_utils"]
            else (
                f"{kernel_su} DIFFERS from pin {PINNED['summit_utils']}"
                if re.fullmatch(r"w\.\d{4}\.\d{2}", kernel_su)
                else f"{kernel_su} (an EUPS version, not comparable with the weekly tag {PINNED['summit_utils']}; "
                "the effective-wavelength and guider seeing module rows are the binding checks)"
            )
        )
    ),
    "kernel: effective wavelengths": check_effective_wavelengths(sel),
    "kernel: guider seeing module": check_guider_seeing_module(),
    "producer: Rapid Analysis versions (all selected visits)": ra_text,
    "producer: Rapid Analysis versions (visits with guider seeing)": guider_text
    + "; "
    + guider_status,
    "producer: guider outer scale": l0_text,
    "producer: RINGSS ESS CSC (salIndex 304)": ess_text
    + "; "
    + ess_status
    + ". The RINGSS reduction code and the visit1_efd transform are not versioned anywhere queryable",
}
provenance = {
    "method version": METHOD_VERSION,
    "run (UTC)": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "PSF column": "cdb_lsstcam.visit1_quicklook.psf_sigma_median (Summit Rapid Analysis; producer versions in the "
    "'producer: Rapid Analysis versions' row), as delivered",
    "RINGSS": ringss_description,
    "DIMM": "cdb_lsstcam.visit1.dimm_seeing"
    + (
        f"; only visits with visit1.wind_speed >= {dimm_min_wind_speed:g} m/s"
        if dimm_min_wind_speed > 0
        else "; no wind-speed cut"
    ),
    "outer scale (RINGSS and DIMM)": (
        f"von Karman outer-scale correction (Tokovinin 2002), L0 = {outer_scale_m:g} m"
        if outer_scale_m
        else "off (Kolmogorov FWHM)"
    )
    + ", evaluated at the visit band and airmass (r0 scaled by (lambda/500)^1.2 X^-0.6 first)",
    "guider": "sqrt(guider_mid_layer_seeing^2 + guider_free_seeing^2) from cdb_lsstcam.visit1_quicklook "
    "(lsst.summit.utils.guiders.seeing via rubintv_production), used as stored",
    "monitor to visit frame": "; ".join(f"{m.tag}: {m.conversion}" for m in MONITORS),
    "effective wavelengths": EFFECTIVE_WAVELENGTH_SOURCE,
    **pin_rows,
    "ConsDB endpoint": consdb.url,
    "kernel image": os.environ.get("JUPYTER_IMAGE_SPEC", "not a Nublado pod"),
    "image description": os.environ.get("IMAGE_DESCRIPTION", "n/a"),
    "stack": stack_version(),
    "python / numpy / pandas": f"{platform.python_version()} / {np.__version__} / {pd.__version__}",
}
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame.from_dict(provenance, orient="index", columns=["value"]))